In [1]:
import sys; sys.path.append("../src")  # make ../src importable
import numpy as np, pandas as pd, tensorflow as tf  # numpy: arrays; pandas: tables; TensorFlow: deep learning
from tensorflow.keras import layers, models  # Keras building blocks
import pipeline  # shared data module (split, labels, tf.data pipelines)

SEED = 42  # fixed seed, as in the other notebooks
tf.keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow together
y = pipeline.val_df["label"].values  # true validation classes
train_ds = pipeline.make_dataset(pipeline.train_df, shuffle=True, augment_data=True)  # training data: shuffled + augmented
val_ds = pipeline.make_dataset(pipeline.val_df, shuffle=False, augment_data=False)  # validation data: fixed order, no augmentation
errs = lambda q: int((q.argmax(axis=1) != y).sum())  # helper: validation errors of a probability table
print("Train:", len(pipeline.train_df), "| Val:", len(pipeline.val_df))  # expect 50928 and 12732

I0000 00:00:1792567225.093358     805 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792567225.136308     805 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792567225.957876     805 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792567227.225328     805 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7535 MB mem

Train: 50928 | Val: 12732


In [2]:
def train_backbone(tag, make_base, seed=42, unfreeze_frac=0.2):  # two-stage training of one backbone; saves its probabilities
    tf.keras.utils.set_random_seed(seed)  # seed for this run
    base = make_base()  # the pretrained backbone (ImageNet weights, no classifier head)
    base.trainable = False  # stage 1: frozen backbone, train only the new head
    inputs = layers.Input(shape=(224, 224, 3))  # model input: 224 x 224 RGB, values in [0,1]
    x = layers.Rescaling(255.0)(inputs)  # these backbones expect 0-255 pixels (their own preprocessing is built in)
    x = base(x, training=False)  # frozen backbone in inference mode
    x = layers.GlobalAveragePooling2D()(x)  # average each feature map to one number
    x = layers.Dropout(0.3)(x)  # same dropout as the other models
    outputs = layers.Dense(8, activation="softmax")(x)  # 8 class probabilities
    model = models.Model(inputs, outputs)  # assemble the full model
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),  # stage 1 learning rate 1e-3
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])  # same loss and metric as notebook 03
    model.fit(train_ds, validation_data=val_ds, epochs=5, verbose=2)  # stage 1: 5 epochs

    base.trainable = True  # stage 2: allow the backbone to change...
    cut = int(len(base.layers) * (1 - unfreeze_frac))  # ...but only its last 20% of layers
    for i, layer in enumerate(base.layers):  # go through the backbone layer by layer
        layer.trainable = (i >= cut) and not isinstance(layer, layers.BatchNormalization)  # last layers only; BatchNorm stays frozen
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),  # ten times lower learning rate for fine-tuning
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])  # recompile after changing trainable
    cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),  # stop early, keep best weights
          tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1, min_lr=1e-6, verbose=1)]  # lower the learning rate when stalled
    model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb, verbose=2)  # stage 2: up to 12 epochs

    pv = model.predict(val_ds, verbose=0)  # validation probabilities
    pt = model.predict(pipeline.make_test_dataset(), verbose=0)  # test probabilities (predictions only)
    np.save(f"../models/new_{tag}_val.npy", pv); np.save(f"../models/new_{tag}_test.npy", pt)  # save both for the ensembles
    print(f"{tag}: {errs(pv)} validation errors | accuracy {1 - errs(pv) / len(y):.4f}")  # result of this model
    return pv, pt  # hand the probabilities back

In [3]:
from tensorflow.keras.applications import EfficientNetV2S  # a newer EfficientNet family member
pv_v2s, pt_v2s = train_backbone(  # train and evaluate EfficientNetV2-S
    "effv2s", lambda: EfficientNetV2S(include_top=False, weights="imagenet", input_shape=(224, 224, 3)))

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/5


/home/suemon/enge707-kaggle/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1792567245.298167     921 service.cc:153] XLA service 0x7227ec0a9d80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1792567245.298207     921 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3080, Compute Capability 8.6 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1792567246.065248     921 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1792567248.205081     921 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1792567248.977998     921 dot_merger.cc:481] Merg

1592/1592 - 119s - 75ms/step - accuracy: 0.8342 - loss: 0.4985 - val_accuracy: 0.9175 - val_loss: 0.2767
Epoch 2/5
1592/1592 - 63s - 40ms/step - accuracy: 0.8917 - loss: 0.3073 - val_accuracy: 0.9315 - val_loss: 0.2229
Epoch 3/5
1592/1592 - 62s - 39ms/step - accuracy: 0.8959 - loss: 0.2838 - val_accuracy: 0.9322 - val_loss: 0.2063
Epoch 4/5
1592/1592 - 62s - 39ms/step - accuracy: 0.9016 - loss: 0.2695 - val_accuracy: 0.9365 - val_loss: 0.1934
Epoch 5/5
1592/1592 - 62s - 39ms/step - accuracy: 0.9012 - loss: 0.2630 - val_accuracy: 0.9372 - val_loss: 0.1852
Epoch 1/12


I0000 00:00:1792567620.282817     922 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_115720__.489
I0000 00:00:1792567703.429081     921 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_115720__.489


1592/1592 - 135s - 85ms/step - accuracy: 0.9590 - loss: 0.1105 - val_accuracy: 0.9819 - val_loss: 0.0462 - learning_rate: 1.0000e-04
Epoch 2/12

Epoch 2: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
1592/1592 - 78s - 49ms/step - accuracy: 0.9817 - loss: 0.0503 - val_accuracy: 0.9801 - val_loss: 0.0580 - learning_rate: 1.0000e-04
Epoch 3/12
1592/1592 - 78s - 49ms/step - accuracy: 0.9896 - loss: 0.0284 - val_accuracy: 0.9891 - val_loss: 0.0298 - learning_rate: 3.0000e-05
Epoch 4/12

Epoch 4: ReduceLROnPlateau reducing learning rate to 8.999999772640877e-06.
1592/1592 - 77s - 49ms/step - accuracy: 0.9917 - loss: 0.0231 - val_accuracy: 0.9870 - val_loss: 0.0351 - learning_rate: 3.0000e-05
Epoch 5/12
1592/1592 - 75s - 47ms/step - accuracy: 0.9934 - loss: 0.0180 - val_accuracy: 0.9904 - val_loss: 0.0277 - learning_rate: 9.0000e-06
Epoch 6/12
1592/1592 - 76s - 48ms/step - accuracy: 0.9941 - loss: 0.0170 - val_accuracy: 0.9899 - val_loss: 0.0272 - learning_rate: 9.0000e-

In [4]:
from tensorflow.keras.applications import ConvNeXtTiny  # a modern convolutional network (different design from EfficientNet/ResNet)
pv_cnx, pt_cnx = train_backbone(  # train and evaluate ConvNeXt-Tiny
    "convnext", lambda: ConvNeXtTiny(include_top=False, weights="imagenet", input_shape=(224, 224, 3)))

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/5


I0000 00:00:1792568427.740773     921 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_251098__.346
E0000 00:00:1792568429.042628     921 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1792568429.625831   13357 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_150', 9288 bytes spill stores, 9116 bytes spill loads

I0000 00:00:1792568430.617089   13348 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_162', 9224 bytes spill stores, 9016 bytes spill loads

I0000 00:00:1792568431.075328   13358 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_210', 8756 bytes spill stores, 8748 bytes spill loads

I0000 00:00:1792568431.676542   13352 subpr

1592/1592 - 97s - 61ms/step - accuracy: 0.8317 - loss: 0.5069 - val_accuracy: 0.9239 - val_loss: 0.2439
Epoch 2/5
1592/1592 - 60s - 38ms/step - accuracy: 0.9049 - loss: 0.2770 - val_accuracy: 0.9391 - val_loss: 0.1885
Epoch 3/5
1592/1592 - 61s - 38ms/step - accuracy: 0.9116 - loss: 0.2452 - val_accuracy: 0.9405 - val_loss: 0.1728
Epoch 4/5
1592/1592 - 61s - 38ms/step - accuracy: 0.9170 - loss: 0.2284 - val_accuracy: 0.9460 - val_loss: 0.1609
Epoch 5/5
1592/1592 - 60s - 38ms/step - accuracy: 0.9193 - loss: 0.2208 - val_accuracy: 0.9511 - val_loss: 0.1456
Epoch 1/12


I0000 00:00:1792568769.097951     923 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_309929__.346
I0000 00:00:1792568769.722830   14569 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_226', 8480 bytes spill stores, 8508 bytes spill loads

I0000 00:00:1792568770.592078   14571 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_1_134', 88 bytes spill stores, 72 bytes spill loads

E0000 00:00:1792568771.065631     923 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1792568771.695800   14581 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_1_138', 56 bytes spill stores, 56 bytes spill loads

I0000 00:00:1792568772.180804   14579 subproces

1592/1592 - 112s - 70ms/step - accuracy: 0.9651 - loss: 0.0983 - val_accuracy: 0.9863 - val_loss: 0.0387 - learning_rate: 1.0000e-04
Epoch 2/12

Epoch 2: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
1592/1592 - 75s - 47ms/step - accuracy: 0.9860 - loss: 0.0381 - val_accuracy: 0.9853 - val_loss: 0.0440 - learning_rate: 1.0000e-04
Epoch 3/12
1592/1592 - 74s - 47ms/step - accuracy: 0.9949 - loss: 0.0163 - val_accuracy: 0.9907 - val_loss: 0.0264 - learning_rate: 3.0000e-05
Epoch 4/12

Epoch 4: ReduceLROnPlateau reducing learning rate to 8.999999772640877e-06.
1592/1592 - 75s - 47ms/step - accuracy: 0.9961 - loss: 0.0121 - val_accuracy: 0.9918 - val_loss: 0.0280 - learning_rate: 3.0000e-05
Epoch 5/12
1592/1592 - 74s - 46ms/step - accuracy: 0.9973 - loss: 0.0086 - val_accuracy: 0.9920 - val_loss: 0.0247 - learning_rate: 9.0000e-06
Epoch 6/12

Epoch 6: ReduceLROnPlateau reducing learning rate to 2.6999998226528985e-06.
1592/1592 - 74s - 46ms/step - accuracy: 0.9982 - lo

I0000 00:00:1792569739.221897   16458 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_100', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1792569739.223904   16452 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_100', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1792569739.402083   16446 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_100', 92 bytes spill stores, 64 bytes spill loads

I0000 00:00:1792569740.010040   16451 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_136', 8748 bytes spill stores, 8732 bytes spill loads



convnext: 97 validation errors | accuracy 0.9924


In [5]:
MAX_VAL_ERRORS = 30  # rule fixed in advance: write a submission only if the ensemble has at most this many validation errors

lv = lambda run, f: np.load(f"../{run}/models/{f}")  # helper: load a saved array from a backup folder
lm = lambda f: np.load(f"../models/{f}")  # helper: load a saved array from the models folder
old = {  # the 8 existing models: (validation probabilities, test probabilities)
    "base_eff1": (lv("backup_run1", "val_probs.npy"), lv("backup_run1", "test_probs.npy")),  # baseline EfficientNetB0, run 1
    "base_r501": (lv("backup_run1", "val_probs_r50.npy"), lv("backup_run1", "test_probs_r50.npy")),  # baseline ResNet50V2, run 1
    "base_eff2": (lv("backup_run2", "val_probs.npy"), lv("backup_run2", "test_probs.npy")),  # baseline EfficientNetB0, rerun
    "base_r502": (lv("backup_run2", "val_probs_r50.npy"), lv("backup_run2", "test_probs_r50.npy")),  # baseline ResNet50V2, rerun
    "reg_eff1": (lm("reg_val_probs.npy"), lm("reg_test_probs.npy")),  # regularised EfficientNetB0, seed 42
    "reg_eff2": (lm("reg2_val_probs.npy"), lm("reg2_test_probs.npy")),  # regularised EfficientNetB0, seed 7
    "reg_r501": (lm("reg_r50_val_probs.npy"), lm("reg_r50_test_probs.npy")),  # regularised ResNet50V2, seed 42
    "reg_r502": (lm("reg_r50b_val_probs.npy"), lm("reg_r50b_test_probs.npy")),  # regularised ResNet50V2, seed 7
}
import os  # os: check which saved files exist
new = {}  # the new models available so far: name -> (validation probabilities, test probabilities)
for tag in ("effv2s", "convnext"):  # the two new backbones
    if os.path.exists(f"../models/new_{tag}_val.npy"):  # only use a backbone that has been trained and saved
        new[tag] = (lm(f"new_{tag}_val.npy"), lm(f"new_{tag}_test.npy"))  # load its saved probabilities
print("New models available:", list(new))  # shows which backbones will be included
assert new, "Train at least one new backbone first (Step 3 or Step 4)"  # stop with a clear message if none exist yet
k = len(new)  # number of new models, used in the candidate names below
allm = {**old, **new}  # every available model
reg4 = ["reg_eff1", "reg_eff2", "reg_r501", "reg_r502"]  # the four regularised models (v4a)
six = ["base_eff1", "base_r501", "base_eff2", "base_r502", "reg_eff1", "reg_eff2"]  # the six models of v4b
candidates = {  # ensembles to compare, as lists of model names
    "v4a (4 regularised) [check: 35]": reg4,  # should reproduce 35 errors
    "v4b (6 models) [check: 45]": six,  # should reproduce 45 errors
    "all 8 existing": list(old),  # every existing model
    f"v4a + {k} new": reg4 + list(new),  # regularised four plus the two new backbones
    f"v4b + {k} new": six + list(new),  # six-model mix plus the two new backbones
    f"all 8 + {k} new": list(allm),  # everything
    "new models only": list(new),  # only the new backbones
}
results = {}  # name -> validation errors
for name, members in candidates.items():  # evaluate each candidate ensemble
    p = sum(allm[m][0] for m in members) / len(members)  # equal-weight average of the members' validation probabilities
    results[name] = errs(p)  # count its validation errors
    print(f"{name:34s} {len(members):2d} models | {results[name]:3d} errors | accuracy {1 - results[name] / len(y):.4f}")  # one line per ensemble

New models available: ['effv2s', 'convnext']
v4a (4 regularised) [check: 35]     4 models |  35 errors | accuracy 0.9973
v4b (6 models) [check: 45]          6 models |  45 errors | accuracy 0.9965
all 8 existing                      8 models |  36 errors | accuracy 0.9972
v4a + 2 new                         6 models |  35 errors | accuracy 0.9973
v4b + 2 new                         8 models |  40 errors | accuracy 0.9969
all 8 + 2 new                      10 models |  37 errors | accuracy 0.9971
new models only                     2 models |  85 errors | accuracy 0.9933


In [6]:
best = min((n for n in candidates if "new" in n or "all 8 +" in n), key=lambda n: results[n])  # best candidate that includes a new model
print("Best candidate with a new model:", best, "->", results[best], "errors")  # show the choice
if results[best] <= MAX_VAL_ERRORS:  # only write a file if the pre-set rule is met
    members = candidates[best]  # members of the chosen ensemble
    t = sum(allm[m][1] for m in members) / len(members)  # equal-weight average of their test probabilities
    sub = pd.DataFrame({"image_id": pipeline.test_df["image_id"].values,  # test ids in Kaggle's order
                        "class": [pipeline.idx_to_class[i] for i in t.argmax(axis=1)]})  # predicted class names
    sub.to_csv("../submissions/submission_v5_ensemble.csv", index=False)  # write the submission file
    sample = pd.read_csv("../data/sample_submission.csv")  # Kaggle's template for the format check
    print("Saved. Shape:", sub.shape, "| columns match:", list(sub.columns) == list(sample.columns),  # shape and column check
          "| IDs match:", (sub["image_id"].values == sample["image_id"].values).all())  # id order check
else:  # rule not met
    print(f"Rule not met (more than {MAX_VAL_ERRORS} errors): no submission file written.")  # nothing is written

Best candidate with a new model: v4a + 2 new -> 35 errors
Rule not met (more than 30 errors): no submission file written.


In [7]:
members = list(allm)  # all 10 models: the 8 existing plus the 2 new backbones
t = sum(allm[m][1] for m in members) / len(members)  # equal-weight average of their test probabilities
sub = pd.DataFrame({"image_id": pipeline.test_df["image_id"].values,  # test ids in Kaggle's order
                    "class": [pipeline.idx_to_class[i] for i in t.argmax(axis=1)]})  # predicted class names
sub.to_csv("../submissions/submission_v5_exploratory_all10.csv", index=False)  # exploratory file; the pre-set rule was not met
sample = pd.read_csv("../data/sample_submission.csv")  # Kaggle's template for the format check
print(len(members), "models | shape:", sub.shape, "| columns match:", list(sub.columns) == list(sample.columns),
      "| IDs match:", (sub["image_id"].values == sample["image_id"].values).all())

10 models | shape: (29443, 2) | columns match: True | IDs match: True
